# TOD Simulation

Pipeline de geração de Time-Ordered Data (TOD) do RadioSkySim.

O notebook demonstra:
- fontes discretas;
- emissão difusa da Via Láctea;
- mapas HEALPix;
- pointing e beam;
- persistência HDF5.


## 1. Setup


In [12]:
"""Imports e setup do ambiente."""

from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import nest_asyncio
import numpy as np  # noqa: F401

# ── Detecta raiz do projeto e adiciona src/ ao path ANTES dos imports do pacote ──
cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents]
project_root = None
for p in candidates:
    if (p / "src" / "radioskysims").exists() and (p / "settings.toml").exists():
        project_root = p
        break

if project_root is None:
    msg = "Raiz do projeto não encontrada."
    raise RuntimeError(msg)

src_path = str((project_root / "src").resolve())
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# ── Imports do projeto (após configurar sys.path) ──
from radioskysims.application.services.sim_orchestrator import SimOrchestrator
from radioskysims.domain.common import TimeRange

nest_asyncio.apply()

## 2. Emissão difusa da Via Láctea


In [ ]:
orch = SimOrchestrator.from_settings("bingo")

tr_galaxy = TimeRange.from_iso(
    "2026-07-26T00:00:00",
    "2026-07-26T00:10:00",
    resolution=60,
)

print("✅ BINGO configurado")


## 3. Geração dos mapas galácticos

A geração dos 381 mapas com PySM3 pode ser custosa.
Por padrão, o notebook reutiliza os mapas HDF5 já persistidos.


In [ ]:
GENERATE_GALAXY_MAPS = False

galaxy_result = None

if GENERATE_GALAXY_MAPS:
    galaxy_result = orch.run(
        constellations=[],
        time_range=tr_galaxy,
        include_local_sources=False,
        include_radio_sources=False,
        include_galaxy=True,
        galaxy_nside=64,
        persist_skymaps=True,
        skymap_overwrite=False,
        use_processor=False,
    )

    print("✅ Mapas Galaxy gerados com PySM3")
else:
    print("ℹ️ Reutilizando mapas HDF5 já persistidos")


## 4. Carregamento dos mapas HDF5

Os mapas HEALPix persistidos são recuperados pelo repository
configurado no Composition Root.


In [ ]:
repository = orch._skymap_repository

if repository is None:
    raise RuntimeError("SkyMap repository não configurado.")

loaded_maps = []

for freq_mhz in orch._ctx.frequency_range.value:
    key = f"galaxy_{freq_mhz:g}mhz_nside64"

    skymap = repository.load(key)

    if skymap is None:
        raise RuntimeError(f"Mapa ausente: {key}")

    loaded_maps.append(skymap)

loaded_maps = tuple(loaded_maps)

print(f"✅ {len(loaded_maps)} mapas carregados")
print(
    "Faixa:",
    loaded_maps[0].freq_ghz,
    "→",
    loaded_maps[-1].freq_ghz,
    "GHz",
)


## 5. Mapas persistidos → TOD

Os mapas carregados do HDF5 são projetados sobre o pointing
temporal do BINGO, com aplicação do beam.


In [ ]:
from radioskysims.application.services.simulation_service import (
    _compute_time_grid,
)
from radioskysims.domain.geometry import compute_pointing_track

time_grid = _compute_time_grid(tr_galaxy)

pointing_track = compute_pointing_track(
    observer=orch._ctx.observer,
    time_grid=time_grid,
    alt_deg=80.0,
    az_deg=30.0,
)

loaded_maps_tod = orch._tod_orchestrator.run_maps(
    maps=loaded_maps,
    pointing_track=pointing_track,
    projector=orch._skymap_tod_projector,
    use_beam=True,
)

print("✅ TOD gerado a partir dos mapas HDF5")
print("dims:", dict(loaded_maps_tod.sizes))
print("variáveis:", list(loaded_maps_tod.data_vars))


## 6. Validação do round-trip

Quando os mapas são gerados novamente com PySM3, o TOD original é
comparado com o TOD reconstruído após persistência e carregamento HDF5.


In [ ]:
import numpy as np

if galaxy_result is not None:
    original_tod = galaxy_result.simulation_result.dataset

    np.testing.assert_allclose(
        loaded_maps_tod["T_antenna"].values,
        original_tod["T_antenna"].values,
        rtol=1e-10,
        atol=1e-12,
    )

    np.testing.assert_allclose(
        loaded_maps_tod["power"].values,
        original_tod["power"].values,
        rtol=1e-10,
        atol=1e-30,
    )

    print("✅ Round-trip HDF5 → SkyMap → TOD validado")
else:
    print("ℹ️ Round-trip não executado nesta execução")


## 7. TOD de fontes discretas

Fluxo tradicional baseado em fontes, fluxos e tracks.


In [ ]:
RUN_DISCRETE_SOURCES = False
comp_result = None

if RUN_DISCRETE_SOURCES:
    # ── Intervalo de observação  ──
    tr_tod = TimeRange.from_iso("2026-07-26T00:00:00", "2026-07-27T00:00:00", resolution=60)

    # ── Orquestrador BINGO ──
    orch = SimOrchestrator.from_settings("bingo")


    # ── Executa simulação multi-fonte (resultado LAZY — NUNCA materializar) ──
    # O dataset simulation_result é sempre muito grande e não pode ser computado
    # inteiro de uma vez. O orquestrador retorna arrays dask; events e to_hdf5/
    # to_zarr consomem o dataset por chunks ao longo do tempo (memória limitada
    # ao tamanho do chunk).
    comp_result = orch.run(
        constellations=["GPS", "GALILEO", "GLONASS", "BEIDOU"],
        time_range=tr_tod,
        include_local_sources=True,  # Sol
        include_radio_sources=False,  # NVSS (se houver dados)
        use_processor=True,
    )

    # ds_orch permanece LAZY (dask). O resumo (summary/format_summary) é
    # lazy-safe — preenche dims/chunks/tamanho sem computar o dataset.
    import dask.array as _da

    ds_orch = comp_result.simulation_result.dataset
    print(f"ℹ️  dims={dict(ds_orch.sizes)} | "
          f"lazy={'dask' if isinstance(ds_orch['T_antenna'].data, _da.Array) else 'numpy'}")
    print(comp_result.format_summary())
else:
    print("ℹ️ TOD de fontes discretas não executado")


## 8. Persistência do SimulationResult

Persistência do resultado da simulação em HDF5 e Zarr.


## 9. Repositório de simulações — opcional

Operações de registro, carregamento e remoção no repositório.


## 10. Eventos e trânsitos

Extração dos eventos associados às fontes que atravessam o beam.


In [ ]:
if comp_result is not None:
    print("include_galaxy:", galaxy_result.config.get("include_galaxy"))
    print("persist_skymaps:", galaxy_result.config.get("persist_skymaps"))
    print("galaxy_nside:", galaxy_result.config.get("galaxy_nside"))
else:
    print("ℹ️ Eventos não executados")
